# Td base 特征生产：4 个量化列 + Mordred 矩阵

全部从原始数据现算：

- `TDT/data/fixed.xlsx`：1426 行，Td 标签 + SMILES
- `Parents/parent_XXXX/wbo`：xTB 的 Wiberg 键级 → `max_wbo`（重原子键里的最大值）
- `xtb_work/BDE/BDE_XXXX/{C-NO2,N-NO2,O-NO2}/`：断键单点能 json → 三列垂直 BDE（没有那种键的分子留 NaN）
- Mordred 2D 全量：从 SMILES 现算，缺值率 > 30% 的列删掉、其余 NaN 填 0

产出：`TDT/out/features_1426.npz`（`mmat` / `mcols` / `qx` / `y`），给 `test.ipynb` 读。

耗时：本机实测约 1 分钟（Mordred 现算 + 4 个量化列现算，都不重跑 xTB）。

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from mordred import Calculator, descriptors
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")

In [2]:
import sys

#仓库根：往上找含 config.py 的目录（本 notebook 就在仓库里，换机器不用改）
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

In [3]:
N_MOL = 1426
HARTREE = 2625.5
QUANT_COLS = ["max_wbo", "C-NO2", "N-NO2", "O-NO2"]
BDE_NAMES = ["C-NO2", "N-NO2", "O-NO2"]
OUT = config.OUT      #产物目录 TDT/out/（已写进 .gitignore，不进版本库）
OUT.mkdir(exist_ok = True)

In [4]:
# 主表：1426 行，Td 标签 + SMILES（前两列）
df = pd.read_excel(config.DATA_TDT / "fixed.xlsx")
df.columns = ["Td", "SMILES"]
y = df["Td"].values.astype(float)
df.head(5)

,Td,SMILES
0,225.0,O=[N+]([O-])c1cc([N+](=O)[O-])n(-c2nonc2[N+](=...
1,241.0,Nn1cc([N+](=O)[O-])c([N+](=O)[O-])n1
2,136.0,O=[N+]([O-])N1CN=C(C([N+](=O)[O-])([N+](=O)[O-...
3,174.0,[N-]=[N+]=Nc1nc([N+](=O)[O-])n[nH]1
4,141.0,O=[N+]([O-])c1ncn(NCC([N+](=O)[O-])([N+](=O)[O...


In [5]:
# 读取 wbo 文件，返回该分子中所有键的最大 wbo 值（如果没有 wbo 文件或没有键，则返回 np.nan）
def read_wbo_max(i, smiles):
    f = config.PARENTS / f"parent_{i:04d}" / "wbo"
    if not f.exists():
        return np.nan
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    wbo = pd.read_csv(f, sep = r"\s+", names = ["a", "b", "wbo"])
    wbo_d = {(min(a, b) - 1, max(a, b) - 1): w for a, b, w in wbo.values}
    vals = []
    for b in mol.GetBonds():
        if b.GetBeginAtom().GetAtomicNum() < 2 or b.GetEndAtom().GetAtomicNum() < 2:
            continue
        k = (min(b.GetBeginAtomIdx(), b.GetEndAtomIdx()), max(b.GetBeginAtomIdx(), b.GetEndAtomIdx()))
        w = wbo_d.get(k)
        if w is not None:
            vals.append(w)
    return max(vals) if vals else np.nan

max_wbo = np.array([read_wbo_max(i, s) for i, s in enumerate(df["SMILES"])], dtype = float)
int(np.isnan(max_wbo).sum())

0

In [6]:
# 三列垂直 BDE：读现成的 parent / fragA_u1 / fragB_u1 单点能，(E_A + E_B - E_parent) * 2625.5
def read_energy(path):
    text = path.read_text(encoding = "utf-8")
    return float(re.search(r'"total energy"\s*:\s*(-?[\d.eE+-]+)', text).group(1))

def bde_of_bond(folder):
    ep = read_energy(folder / "parent.json")
    e = {name: read_energy(folder / f"{name}_u1.json") for name in ("fragA", "fragB")}
    return (e["fragA"] + e["fragB"] - ep) * HARTREE

def read_bde(i):
    sub = config.XTB_WORK / "BDE" / f"BDE_{i:04d}"
    vals = []
    for name in BDE_NAMES:
        folder = sub / name
        try:
            vals.append(bde_of_bond(folder) if folder.exists() else np.nan)
        except Exception:
            vals.append(np.nan)
    return vals

bde = np.array([read_bde(i) for i in range(N_MOL)], dtype = float)
qx = np.column_stack([max_wbo, bde])      #4 列：max_wbo + C-/N-/O-NO2
bad = [name for j, name in enumerate(QUANT_COLS) if np.isnan(qx[:, j]).all()]
if bad:
    raise RuntimeError("量化列全 NaN：" + "、".join(bad) + "。检查 CHEM_PARENTS / CHEM_XTB_WORK 是否指对。")
pd.Series(np.isnan(qx).sum(axis = 0), index = QUANT_COLS)

max_wbo       0
C-NO2       463
N-NO2      1008
O-NO2      1346
dtype: int64

In [7]:
# Mordred 2D 全量：从 SMILES 现算，缺值率 > 30% 的列删掉，其余 NaN 填 0
mols = [Chem.AddHs(Chem.MolFromSmiles(s)) for s in df["SMILES"]]
md = Calculator(descriptors, ignore_3D = True).pandas(mols, nproc = 8, quiet = False)
md = md.apply(pd.to_numeric, errors = "coerce")
md = md.loc[:, md.notna().mean() >= 0.7]
mmat = md.fillna(0).values.astype(np.float32)
mcols = list(md.columns)
mmat.shape

100%|██████████| 1426/1426 [00:21<00:00, 66.50it/s]


(1426, 1449)

In [8]:
# 四份数据必须同长同序（fixed.xlsx 的 SMILES / Parents / BDE / Mordred 都按 id 排好）
assert len(df) == len(qx) == len(mmat) == N_MOL, (len(df), len(qx), len(mmat))
assert not np.isnan(y).any()

# 出处写进 npz：以后拿到 features_1426.npz 能直接看出是哪次、用了哪些原始目录
meta = np.array([f"生成时间 {pd.Timestamp.now():%Y-%m-%d %H:%M:%S}",
                 f"标签 {config.DATA_TDT / 'fixed.xlsx'}",
                 f"wbo {config.PARENTS}",
                 f"BDE {config.XTB_WORK / 'BDE'}"], dtype = object)
np.savez(OUT / "features_1426.npz", mmat = mmat, mcols = np.array(mcols, dtype = object), qx = qx, y = y, meta = meta)
print(f"已保存 {OUT / 'features_1426.npz'}")
print("  " + "\n  ".join(meta))

已保存 C:\Users\czywg\Desktop\Programmer_czy\chem_project\TDT\out\features_1426.npz
  生成时间 2026-11-10 11:18:17
  标签 C:\Users\czywg\Desktop\Programmer_czy\chem_project\TDT\data\fixed.xlsx
  wbo C:\Users\czywg\Desktop\Programmer_czy\xtb_work\Parents
  BDE C:\Users\czywg\Desktop\Programmer_czy\xtb_work\BDE
